# DAG-SA v3 — confirmatory evaluation

This notebook runs the whole study end to end, unattended, and is safe to
re-run after a Colab disconnect: every completed unit is appended to
`predictions.jsonl` on Drive and skipped on the next run.

**What it produces**

1. `preregistration.json` — the protocol, frozen before the run
2. `predictions.jsonl` — one line per (dataset, subject, seed) unit, holding
   every method's predictions on the identical test trials
3. `report.md` / `report.json` — the comparison table (mean ± sd, 95% CI,
   Cohen's kappa), per-unit paired McNemar with a significance-gated
   win/tie/loss count, per-subject and class-wise metrics, the ablation, and
   the verdict against the pre-registered success criterion
4. `run.log` — a full log

**Before running**, set `DRIVE_ROOT` below. Datasets are expected at
`MyDrive/EEG_DAGSA/dataset` (BCI IV Dataset 1) and `MyDrive/EEG_DAGSA/dataset_2a`
(Dataset 2a).

**Runtime.** Everything except EEGNet is CPU-bound and takes roughly 40 minutes
for all three datasets. EEGNet dominates the rest; on a T4 the full study is
about 3–5 hours. Choose a GPU runtime.


In [ ]:
# ---- configuration ------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/EEG_DAGSA"
OUT_DIR    = f"{DRIVE_ROOT}/results_v3"
DS1_DIR    = f"{DRIVE_ROOT}/dataset"
DS2A_DIR   = f"{DRIVE_ROOT}/dataset_2a"

RUN_EEGNET   = True     # needs a GPU runtime to be practical
RUN_ABLATION = True
DATASETS     = ("ds2a_binary", "ds1", "ds2a_4class")

import os, sys, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("not on Colab or Drive already mounted:", e)

os.makedirs(OUT_DIR, exist_ok=True)
CODE_DIR = "/content/dagsa_v3"
os.makedirs(CODE_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)
print("output ->", OUT_DIR)


In [ ]:
# ---- dependencies -------------------------------------------------------
# scipy / scikit-learn / numpy are preinstalled on Colab. torch is needed only
# for the EEGNet baseline and is preinstalled on GPU runtimes.
import importlib
for mod, pip in [("numpy", "numpy"), ("scipy", "scipy"),
                 ("sklearn", "scikit-learn"), ("pandas", "pandas")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip],
                       check=True)

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if RUN_EEGNET and not torch.cuda.is_available():
    print("WARNING: no GPU. EEGNet will be very slow. "
          "Runtime > Change runtime type > GPU.")


## Source\n\nThe modules are written out verbatim so the notebook is self-contained.

In [ ]:
import json
_src = json.loads('"\\"\\"\\"\\ndata.py -- clean loaders for BCI Competition IV Dataset 1 and Dataset 2a.\\n\\nWritten fresh for the v3 redesign. Two deliberate departures from the\\nrepository\'s `datasets_io.py`:\\n\\n  * ds2a uses the **documented** motor-imagery window (2.0, 6.0) s relative to\\n    trial onset. The repo\'s driver silently used (0.5, 4.5), which covers 1.5 s\\n    of fixation plus the cue-evoked response plus only ~1.5 s of imagery.\\n    See AUDIT_2026-07-29.md finding A3.\\n  * Epoching returns *unfiltered* epochs; every consumer band-passes for\\n    itself, so no method can accidentally be handed broadband data while\\n    another gets an optimised pass-band (audit finding A1).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Tuple, List\\n\\nimport numpy as np\\nimport scipy.io as sio\\n\\n# --------------------------------------------------------------------------- #\\n# Dataset 1  (BCICIV_calib_ds1*.mat)\\n# --------------------------------------------------------------------------- #\\n# Subjects a, b, f, g are recorded from human subjects.\\n# Subjects c, d, e are ARTIFICIALLY GENERATED by the competition organisers.\\nDS1_REAL = [\\"a\\", \\"b\\", \\"f\\", \\"g\\"]\\nDS1_ARTIFICIAL = [\\"c\\", \\"d\\", \\"e\\"]\\nDS1_ALL = [\\"a\\", \\"b\\", \\"c\\", \\"d\\", \\"e\\", \\"f\\", \\"g\\"]\\n\\n\\ndef load_ds1(dataset_dir, subject: str,\\n             window: Tuple[float, float] = (0.5, 4.0)) -> dict:\\n    \\"\\"\\"\\n    Load one Dataset 1 calibration subject.\\n\\n    Returns dict with X (trials, channels, samples), y (0/1), fs, ch_names.\\n\\n    The calibration files are 100 Hz, 59 channels, 200 cued trials,\\n    two classes drawn per subject from {left hand, right hand, foot}.\\n    Cue onset is at `mrk.pos`; the imagery period is 4 s. We take\\n    (0.5, 4.0) s to skip the cue-evoked transient -- a fixed choice made\\n    before any test data was touched.\\n    \\"\\"\\"\\n    path = Path(dataset_dir) / f\\"BCICIV_calib_ds1{subject}.mat\\"\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"[ds1] missing {path}\\")\\n    m = sio.loadmat(str(path))\\n\\n    # cnt is int16 scaled by 0.1 uV per the dataset description.\\n    cnt = m[\\"cnt\\"].astype(np.float64) * 0.1          # (samples, channels)\\n    nfo = m[\\"nfo\\"]\\n    fs = int(np.asarray(nfo[\\"fs\\"][0, 0]).ravel()[0])\\n    ch_names = [str(c[0]) for c in np.asarray(nfo[\\"clab\\"][0, 0]).ravel()]\\n    classes = [str(np.asarray(c).ravel()[0])\\n               for c in np.asarray(nfo[\\"classes\\"][0, 0]).ravel()]\\n\\n    mrk = m[\\"mrk\\"]\\n    pos = np.asarray(mrk[\\"pos\\"][0, 0]).ravel().astype(int)\\n    ycue = np.asarray(mrk[\\"y\\"][0, 0]).ravel().astype(int)   # -1 / +1\\n\\n    X, y = _epoch(cnt, pos, (ycue > 0).astype(int), fs, window)\\n    return dict(X=X, y=y, fs=fs, ch_names=ch_names,\\n                class_names=classes, dataset=\\"ds1\\", subject=subject)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Dataset 2a  (A0?T.mat, the BBCI/Kaggle `data` cell export)\\n# --------------------------------------------------------------------------- #\\nDS2A_ALL = [1, 2, 3, 4, 5, 6, 7, 8, 9]\\n_DS2A_CLASS_NAMES = [\\"left_hand\\", \\"right_hand\\", \\"feet\\", \\"tongue\\"]\\n\\n\\ndef load_ds2a(dataset_dir, subject: int, variant: str = \\"binary\\",\\n              session: str = \\"T\\",\\n              window: Tuple[float, float] = (2.0, 6.0),\\n              drop_artifacts: bool = False) -> dict:\\n    \\"\\"\\"\\n    Load one Dataset 2a subject.\\n\\n    `window` is seconds relative to the value in the `trial` field, which\\n    marks FIXATION onset. The cue appears at ~2 s and imagery runs ~3-6 s,\\n    so (2.0, 6.0) is the standard 4 s motor-imagery period.\\n\\n    variant: \'binary\' -> left vs right hand (classes 1,2) as labels 0/1\\n             \'4class\' -> all four classes as labels 0..3\\n    \\"\\"\\"\\n    path = Path(dataset_dir) / f\\"A0{int(subject)}{session}.mat\\"\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"[ds2a] missing {path}\\")\\n    m = sio.loadmat(str(path))\\n    runs = m[\\"data\\"]\\n\\n    Xs, ys = [], []\\n    fs = None\\n    for i in range(runs.shape[1]):\\n        r = runs[0, i]\\n        try:\\n            sig = np.asarray(r[\\"X\\"][0, 0], dtype=np.float64)\\n            trial = np.asarray(r[\\"trial\\"][0, 0]).ravel().astype(int)\\n            lab = np.asarray(r[\\"y\\"][0, 0]).ravel().astype(int)\\n            rfs = int(np.asarray(r[\\"fs\\"][0, 0]).ravel()[0])\\n            art = np.asarray(r[\\"artifacts\\"][0, 0]).ravel().astype(int)\\n        except Exception:\\n            continue\\n        # The first runs of 2a are eye-movement / baseline recordings with no\\n        # MI trials; they carry 0 trials or no labels.\\n        if trial.size == 0 or lab.size == 0 or lab.max() == 0:\\n            continue\\n        fs = rfs\\n        # Drop the 3 EOG channels, keep 22 EEG.\\n        sig = sig[:, :22]\\n        # A handful of 2a exports carry NaNs at run boundaries.\\n        if np.isnan(sig).any():\\n            sig = np.nan_to_num(sig, nan=0.0)\\n        Xr, yr = _epoch(sig, trial, lab - 1, rfs, window)\\n        if drop_artifacts and art.size == lab.size:\\n            keep = art[: len(yr)] == 0\\n            Xr, yr = Xr[keep], yr[keep]\\n        Xs.append(Xr)\\n        ys.append(yr)\\n\\n    if not Xs:\\n        raise ValueError(f\\"[ds2a] no MI runs found in {path}\\")\\n    X = np.concatenate(Xs, axis=0)\\n    y = np.concatenate(ys, axis=0)\\n\\n    if variant == \\"binary\\":\\n        keep = np.isin(y, [0, 1])\\n        X, y = X[keep], y[keep].astype(int)\\n        class_names = _DS2A_CLASS_NAMES[:2]\\n    elif variant == \\"4class\\":\\n        class_names = list(_DS2A_CLASS_NAMES)\\n    else:\\n        raise ValueError(f\\"unknown variant {variant!r}\\")\\n\\n    ch_names = [f\\"EEG{i+1}\\" for i in range(X.shape[1])]\\n    return dict(X=X, y=y, fs=fs, ch_names=ch_names,\\n                class_names=class_names, dataset=f\\"ds2a_{variant}\\",\\n                subject=int(subject))\\n\\n\\n# --------------------------------------------------------------------------- #\\ndef _epoch(sig: np.ndarray, onsets: np.ndarray, labels: np.ndarray,\\n           fs: int, window: Tuple[float, float]):\\n    \\"\\"\\"\\n    sig: (samples, channels). Returns (trials, channels, samples), labels.\\n    Trials whose window falls outside the recording are dropped, and the\\n    number dropped is reported by the caller via the shape difference.\\n    \\"\\"\\"\\n    a = int(round(window[0] * fs))\\n    b = int(round(window[1] * fs))\\n    n = b - a\\n    out, lab = [], []\\n    for o, l in zip(onsets, labels):\\n        s, e = o + a, o + b\\n        if s >= 0 and e <= sig.shape[0]:\\n            out.append(sig[s:e, :].T)\\n            lab.append(l)\\n    if not out:\\n        raise ValueError(\\"no valid epochs -- check window/onsets\\")\\n    return np.asarray(out, dtype=np.float64), np.asarray(lab, dtype=int)\\n\\n\\ndef load_unit(dataset: str, dataset_dir, subject) -> dict:\\n    \\"\\"\\"Dispatch used by the runner.\\"\\"\\"\\n    if dataset == \\"ds1\\":\\n        return load_ds1(dataset_dir, str(subject))\\n    if dataset == \\"ds2a_binary\\":\\n        return load_ds2a(dataset_dir, int(subject), variant=\\"binary\\")\\n    if dataset == \\"ds2a_4class\\":\\n        return load_ds2a(dataset_dir, int(subject), variant=\\"4class\\")\\n    raise ValueError(f\\"unknown dataset {dataset!r}\\")\\n\\n\\ndef subjects_for(dataset: str) -> List:\\n    if dataset == \\"ds1\\":\\n        return list(DS1_ALL)\\n    return list(DS2A_ALL)\\n"')
with open(f'{CODE_DIR}/data.py', 'w') as f:
    f.write(_src)
print('wrote data.py', len(_src), 'chars')


In [ ]:
import json
_src = json.loads('"\\"\\"\\"\\npipeline.py -- the v3 proposed method and its building blocks.\\n\\nMETHOD: ARTS  (Aligned Riemannian Transfer Stacking)\\n\\nMotivation, stated so the design is auditable against the diagnosis:\\n\\n  The published DAG-SA ranks ~4e11 ensemble topologies by accuracy on ~30\\n  held-out validation trials. That objective has a standard error of about\\n  9 accuracy points and a resolution of 1/30 = 3.3 points, so the argmax over\\n  a combinatorial space is dominated by selection noise. Re-running the same\\n  method with a different RNG trajectory moves a unit by 17 points (sd).\\n\\n  ARTS removes the three things that produce that noise, one at a time:\\n\\n    1. NO DISCRETE SEARCH. Ensemble construction becomes a convex weighting\\n       problem with a closed, regularised solution instead of an argmax over\\n       a combinatorial space. Given the split, ARTS is deterministic: the only\\n       stochasticity left is the inner-fold assignment.\\n    2. A SELECTION SIGNAL THAT IS THE WHOLE TRAINING SET. Fusion weights are\\n       fit on out-of-fold predictions across all n_train trials (140-230),\\n       not on a 30-trial holdout. Resolution improves from 1/30 to 1/n_train\\n       and there is no held-out split to overfit.\\n    3. A POOL OF GENUINELY DIFFERENT LEARNERS. The published pool is ~540\\n       near-duplicate CSP/CSSP log-variance views over 4 bands; a committee of\\n       four of them is close to a committee of one. ARTS uses one strong\\n       learner per frequency band (Riemannian tangent space), plus, for each\\n       band, a second learner trained on every OTHER subject\'s data after\\n       Euclidean alignment. Members differ by band and by data source, which\\n       is where ensemble gain actually comes from.\\n\\n  Component 3\'s transfer half is also the only component with a documented\\n  effect size large enough to clear the ~4-point resolution bar of a\\n  subject-level comparison (He & Wu 2020, Euclidean alignment for transfer).\\n\\nEverything below is fit on training data only. Per-trial band-pass filtering\\nand per-trial covariance estimation involve no cross-trial statistics, so\\ncovariances may be precomputed for all trials before splitting without\\nleakage; every quantity that IS fitted (the alignment reference, the tangent\\nmap, every classifier, the meta-learner) sees training indices only.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom typing import Dict, List, Optional, Sequence, Tuple\\n\\nimport numpy as np\\nfrom scipy.linalg import eigh\\nfrom scipy.signal import butter, filtfilt\\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.model_selection import StratifiedKFold\\n\\n# --------------------------------------------------------------------------- #\\n# Filter bank\\n# --------------------------------------------------------------------------- #\\n# Six overlapping bands spanning the mu/beta range. Fixed before any test data\\n# was examined; chosen to match the standard FBCSP 4 Hz grid plus a wideband\\n# 8-30 Hz member so that the single-band Riemannian baseline is nested inside\\n# the bank (which is what makes the ablation interpretable).\\nDEFAULT_BANDS: Tuple[Tuple[float, float], ...] = (\\n    (4.0, 8.0), (8.0, 12.0), (12.0, 16.0),\\n    (16.0, 22.0), (22.0, 30.0), (8.0, 30.0),\\n)\\nRIEMANNIAN_BASELINE_BAND = (8.0, 30.0)\\n\\n\\ndef bandpass(X: np.ndarray, low: float, high: float, fs: int,\\n             order: int = 4) -> np.ndarray:\\n    \\"\\"\\"\\n    Zero-phase Butterworth band-pass over the last axis of (trials, ch, time).\\n\\n    filtfilt, not lfilter: the repository used a causal `lfilter`, which\\n    imposes a frequency-dependent phase delay that shifts the discriminative\\n    ERD/ERS window differently in each band. That is harmless when one band is\\n    used and harmful when bands are combined.\\n    \\"\\"\\"\\n    nyq = 0.5 * fs\\n    hi = min(high / nyq, 0.99)\\n    lo = max(low / nyq, 1e-4)\\n    b, a = butter(order, [lo, hi], btype=\\"band\\")\\n    pad = min(3 * max(len(a), len(b)), X.shape[-1] - 1)\\n    return filtfilt(b, a, X, axis=-1, padlen=pad)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Covariance, alignment, tangent space\\n# --------------------------------------------------------------------------- #\\ndef oas_cov(X: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"\\n    Oracle-approximating-shrinkage covariance per trial.\\n\\n    X: (trials, channels, samples) -> (trials, channels, channels).\\n    OAS is the right estimator here: n_samples per trial (350-1000) is not\\n    large relative to n_channels (22-59), and an unshrunk sample covariance\\n    is frequently near-singular, which the tangent-space log map cannot\\n    tolerate.\\n    \\"\\"\\"\\n    n_t, n_c, n_s = X.shape\\n    x = X - X.mean(axis=2, keepdims=True)\\n    s = np.einsum(\\"nct,ndt->ncd\\", x, x) / n_s\\n    mu = np.trace(s, axis1=1, axis2=2) / n_c                    # (n_t,)\\n    alpha = (s * s).mean(axis=(1, 2))\\n    num = alpha + mu * mu\\n    den = (n_s + 1.0) * (alpha - (mu * mu) / n_c)\\n    rho = np.where(den <= 0, 1.0, np.minimum(1.0, num / np.where(den == 0, 1, den)))\\n    I = np.eye(n_c)\\n    return (1.0 - rho)[:, None, None] * s + (rho * mu)[:, None, None] * I\\n\\n\\ndef _inv_sqrtm(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:\\n    w, V = eigh(C)\\n    w = np.maximum(w, eps)\\n    return (V * (w ** -0.5)) @ V.T\\n\\n\\ndef _logm_spd(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:\\n    w, V = eigh(C)\\n    w = np.maximum(w, eps)\\n    return (V * np.log(w)) @ V.T\\n\\n\\ndef euclidean_reference(covs: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"\\n    Euclidean-alignment reference: the arithmetic mean covariance\\n    (He & Wu 2020). Must be estimated from TRAINING trials only.\\n    \\"\\"\\"\\n    return covs.mean(axis=0)\\n\\n\\ndef align(covs: np.ndarray, ref: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Whiten by the reference so the aligned mean is the identity.\\"\\"\\"\\n    W = _inv_sqrtm(ref)\\n    return W @ covs @ W          # batched matmul; einsum here is ~30x slower\\n\\n\\n_TRI_CACHE: Dict[int, Tuple[np.ndarray, np.ndarray, np.ndarray]] = {}\\n\\n\\ndef tangent(covs: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"\\n    Tangent-space map at the identity: vec(logm(C)) over the upper triangle\\n    with off-diagonal entries scaled by sqrt(2) so the Euclidean norm of the\\n    vector equals the Frobenius norm of the matrix logarithm.\\n\\n    Valid because `align` has already moved the mean to I. Using the identity\\n    reference (rather than fitting a Riemannian mean) keeps the map fully\\n    deterministic, which matters: it is one of the stochastic elements the\\n    redesign is trying to remove.\\n    \\"\\"\\"\\n    n, c, _ = covs.shape\\n    if c not in _TRI_CACHE:\\n        iu = np.triu_indices(c)\\n        w = np.where(iu[0] == iu[1], 1.0, np.sqrt(2.0))\\n        _TRI_CACHE[c] = (iu[0], iu[1], w)\\n    r, cc, w = _TRI_CACHE[c]\\n    # Batched symmetric eigendecomposition -- an order of magnitude faster\\n    # than looping scipy.linalg.eigh, and numerically identical.\\n    wv, V = np.linalg.eigh(covs)\\n    wv = np.maximum(wv, 1e-12)\\n    # (V * log w) @ V^T as a batched matmul; einsum without optimize= falls\\n    # back to a naive loop and is ~30x slower here.\\n    L = (V * np.log(wv)[:, None, :]) @ np.swapaxes(V, 1, 2)\\n    return L[:, r, cc] * w\\n\\n\\n# --------------------------------------------------------------------------- #\\n# Per-subject, per-band covariance cache\\n# --------------------------------------------------------------------------- #\\n@dataclass\\nclass SubjectBands:\\n    \\"\\"\\"\\n    Covariances for every trial of one subject, one entry per band.\\n\\n    Leak-free to precompute before splitting: band-pass is a per-trial IIR\\n    with no fitted state and OAS covariance is a per-trial statistic. Nothing\\n    here is estimated across trials.\\n    \\"\\"\\"\\n    covs: Dict[Tuple[float, float], np.ndarray]\\n    y: np.ndarray\\n    subject: object\\n\\n    @staticmethod\\n    def build(X: np.ndarray, y: np.ndarray, fs: int, subject,\\n              bands: Sequence[Tuple[float, float]] = DEFAULT_BANDS\\n              ) -> \\"SubjectBands\\":\\n        covs = {}\\n        for b in bands:\\n            covs[b] = oas_cov(bandpass(X, b[0], b[1], fs))\\n        return SubjectBands(covs=covs, y=np.asarray(y), subject=subject)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# The method\\n# --------------------------------------------------------------------------- #\\n@dataclass\\nclass ARTSConfig:\\n    bands: Sequence[Tuple[float, float]] = DEFAULT_BANDS\\n    use_alignment: bool = True         # Euclidean alignment\\n    use_transfer: bool = True          # cross-subject source views\\n    src_mode: str = \\"pooled\\"           # \'pooled\' | \'per_subject\'\\n    src_bands: Optional[Sequence[Tuple[float, float]]] = None\\n    use_csp: bool = False              # CSP + shrinkage-LDA views\\n    fusion: str = \\"stack\\"              # \'stack\' | \'mean\' | \'best\'\\n    inner_folds: int = 5\\n    n_csp: int = 6\\n    C_self: float = 0.1                # L2 strength, subject-specific views\\n    C_src: float = 0.1                 # L2 strength, source views\\n    C_meta: float = 1.0                # L2 strength, meta-learner\\n    max_iter: int = 500\\n\\n\\n# --------------------------------------------------------------------------- #\\n# CSP computed from covariances (no raw signals needed)\\n# --------------------------------------------------------------------------- #\\ndef csp_from_covs(covs: np.ndarray, y: np.ndarray, n_comp: int = 6\\n                  ) -> np.ndarray:\\n    \\"\\"\\"\\n    CSP spatial filters by generalised eigendecomposition of the class-mean\\n    covariances, taking the components whose generalised eigenvalue is\\n    farthest from 0.5 (i.e. most class-discriminative).\\n\\n    Deriving CSP from the covariances the tangent-space views already use costs\\n    nothing extra and, more importantly, guarantees both view families see\\n    exactly the same signal -- so the ablation isolates the INDUCTIVE BIAS\\n    (rank-reduced log-variance vs full-rank tangent space) rather than any\\n    difference in preprocessing.\\n\\n    For >2 classes, filters are stacked one-vs-rest.\\n    \\"\\"\\"\\n    cls = np.unique(y)\\n    if len(cls) > 2:\\n        return np.concatenate(\\n            [csp_from_covs(covs, (y == c).astype(int), n_comp) for c in cls],\\n            axis=0)\\n    A = covs[y == cls[0]].mean(axis=0)\\n    B = covs[y == cls[1]].mean(axis=0)\\n    A = A / np.trace(A)\\n    B = B / np.trace(B)\\n    w, V = eigh(A, A + B)\\n    order = np.argsort(np.abs(w - 0.5))[::-1]\\n    V = V[:, order]\\n    k = max(1, n_comp // 2)\\n    return np.concatenate([V[:, :k], V[:, -k:]], axis=1).T\\n\\n\\ndef csp_logvar_from_covs(W: np.ndarray, covs: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"log-normalised variance of each CSP component, read off the covariance.\\"\\"\\"\\n    v = np.einsum(\\"kc,ncd,kd->nk\\", W, covs, W)\\n    v = np.maximum(v, 1e-12)\\n    return np.log(v / v.sum(axis=1, keepdims=True))\\n\\n\\ndef _fit_lr(Z, y, C, seed, max_iter):\\n    lr = LogisticRegression(C=C, max_iter=max_iter, random_state=seed,\\n                            class_weight=\\"balanced\\")\\n    lr.fit(Z, y)\\n    return lr\\n\\n\\ndef _logodds(P: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"\\n    Meta-features are log-odds, not probabilities.\\n\\n    A probability saturated at 0.99 and one at 0.999 are nearly identical to a\\n    linear meta-learner but differ by a factor of 10 in evidence. Log-odds\\n    keeps the meta-learner linear in the quantity the base learners are\\n    actually additive in.\\n    \\"\\"\\"\\n    P = np.clip(P, 1e-6, 1 - 1e-6)\\n    if P.shape[1] == 2:\\n        return np.log(P[:, 1:2] / P[:, 0:1])\\n    return np.log(P) - np.log(P).mean(axis=1, keepdims=True)\\n\\n\\nclass ARTS:\\n    \\"\\"\\"\\n    Aligned Riemannian Transfer Stacking.\\n\\n    fit(target_bands, train_idx, source_bands) -> self\\n    predict_proba(test_idx) -> (n_test, n_classes)\\n\\n    Views, per band b:\\n        self_b   : LR on the target subject\'s own aligned tangent vectors\\n        src_b    : LR on every other subject\'s aligned tangent vectors\\n                   (only if use_transfer)\\n\\n    Fusion weights are fit by multinomial L2 logistic regression on the\\n    out-of-fold log-odds of every view over the full training set.\\n    \\"\\"\\"\\n\\n    def __init__(self, cfg: ARTSConfig = ARTSConfig(), seed: int = 0,\\n                 feature_cache: Optional[dict] = None):\\n        self.cfg = cfg\\n        self.seed = seed\\n        # Shared across the ARTS configurations evaluated on ONE unit (the\\n        # proposed method plus every ablation row). They differ in fusion,\\n        # transfer and band subset but re-derive the SAME alignment references\\n        # and tangent maps, which is where nearly all the time goes.\\n        # Keyed by (band, use_alignment, exact fitting index set), so a cache\\n        # hit is only possible when the reference is provably identical.\\n        self._fc = feature_cache if feature_cache is not None else {}\\n        self.classes_: Optional[np.ndarray] = None\\n        self.view_names_: List[str] = []\\n        self._src_models: Dict[Tuple[float, float], object] = {}\\n        self._self_models: Dict[Tuple[float, float], object] = {}\\n        self._ref: Dict[Tuple[float, float], np.ndarray] = {}\\n        self._meta = None\\n        self._uniform = False\\n\\n    # -- feature construction ------------------------------------------ #\\n    def _ref_for(self, tb: SubjectBands, band, idx_fit):\\n        \\"\\"\\"Alignment reference, estimated from idx_fit only.\\"\\"\\"\\n        if not self.cfg.use_alignment:\\n            return None\\n        return euclidean_reference(tb.covs[band][idx_fit])\\n\\n    def _aligned(self, tb: SubjectBands, band, ref, idx):\\n        C = tb.covs[band][idx]\\n        return C if ref is None else align(C, ref)\\n\\n    def _cached(self, tb: SubjectBands, band, idx_fit):\\n        \\"\\"\\"\\n        (aligned covariances, tangent vectors) for ALL trials, under a\\n        reference estimated from idx_fit only.\\n\\n        Computing all trials rather than just the ones needed costs a little\\n        extra per call and saves a great deal across the ablation ladder. It\\n        does not weaken the leakage guarantee: the only thing estimated from\\n        data is `ref`, and `ref` sees idx_fit alone.\\n        \\"\\"\\"\\n        key = (band, self.cfg.use_alignment, idx_fit.tobytes())\\n        hit = self._fc.get(key)\\n        if hit is None:\\n            ref = self._ref_for(tb, band, idx_fit)\\n            C = tb.covs[band] if ref is None else align(tb.covs[band], ref)\\n            hit = (ref, C, tangent(C))\\n            self._fc[key] = hit\\n        return hit\\n\\n    def _target_feats(self, tb: SubjectBands, band, idx_fit, idx_apply):\\n        ref = self._ref_for(tb, band, idx_fit)\\n        return tangent(self._aligned(tb, band, ref, idx_apply)), ref\\n\\n    def _target_feats_with_ref(self, tb, band, ref, idx_apply):\\n        return tangent(self._aligned(tb, band, ref, idx_apply))\\n\\n    # -- source (transfer) views --------------------------------------- #\\n    def fit_sources_per_subject(self, source_bands: Sequence[SubjectBands],\\n                                bands: Optional[Sequence] = None):\\n        \\"\\"\\"\\n        One classifier per (source subject, band) instead of one per band over\\n        pooled sources.\\n\\n        Rationale: pooling eight aligned subjects into a single logistic\\n        regression assumes their discriminative patterns superpose. They do\\n        not -- spatial patterns vary substantially between subjects even after\\n        Euclidean alignment. Keeping them separate lets the fusion layer\\n        discover WHICH source subjects resemble the target, which is the\\n        quantity that actually transfers.\\n        \\"\\"\\"\\n        bands = bands or self.cfg.bands\\n        self._src_per = {}\\n        for sb in source_bands:\\n            for band in bands:\\n                C = sb.covs[band]\\n                if self.cfg.use_alignment:\\n                    C = align(C, euclidean_reference(C))\\n                self._src_per[(sb.subject, band)] = _fit_lr(\\n                    tangent(C), sb.y, self.cfg.C_src, self.seed,\\n                    self.cfg.max_iter)\\n        return self\\n\\n    def fit_sources(self, source_bands: Sequence[SubjectBands]):\\n        \\"\\"\\"\\n        Train one classifier per band on every other subject\'s trials.\\n\\n        Each source subject is aligned by its OWN reference, computed from all\\n        of that subject\'s trials. This is not leakage: source subjects are\\n        disjoint from the target subject, and the target\'s test trials are\\n        never involved. It is also why these models are independent of the\\n        target\'s split and can be cached across seeds.\\n        \\"\\"\\"\\n        self._src_models = {}\\n        if not self.cfg.use_transfer or not source_bands:\\n            return self\\n        for band in self.cfg.bands:\\n            Zs, ys = [], []\\n            for sb in source_bands:\\n                C = sb.covs[band]\\n                if self.cfg.use_alignment:\\n                    C = align(C, euclidean_reference(C))\\n                Zs.append(tangent(C))\\n                ys.append(sb.y)\\n            Z = np.concatenate(Zs, 0)\\n            yy = np.concatenate(ys, 0)\\n            self._src_models[band] = _fit_lr(Z, yy, self.cfg.C_src,\\n                                             self.seed, self.cfg.max_iter)\\n        return self\\n\\n    # -- fit ------------------------------------------------------------ #\\n    def fit(self, tb: SubjectBands, train_idx: np.ndarray,\\n            source_bands: Sequence[SubjectBands] = ()):\\n        cfg = self.cfg\\n        y_tr = tb.y[train_idx]\\n        self.classes_ = np.unique(y_tr)\\n        n_cls = len(self.classes_)\\n\\n        if not self._src_models:\\n            self.fit_sources(source_bands)\\n\\n        # ---- view inventory ------------------------------------------ #\\n        # Order matters: column blocks below are addressed by family offset.\\n        self.view_names_ = [f\\"self_{b[0]:g}-{b[1]:g}\\" for b in cfg.bands]\\n        self._has_src = bool(cfg.use_transfer and self._src_models)\\n        self._has_srcp = bool(cfg.use_transfer and cfg.src_mode == \\"per_subject\\"\\n                              and getattr(self, \\"_src_per\\", None))\\n        n_b = len(cfg.bands)\\n        self._off_src = None\\n        if self._has_src:\\n            self._off_src = len(self.view_names_)\\n            self.view_names_ += [f\\"src_{b[0]:g}-{b[1]:g}\\" for b in cfg.bands]\\n        self._srcp_keys = []\\n        if self._has_srcp:\\n            sbands = cfg.src_bands or cfg.bands\\n            self._off_srcp = len(self.view_names_)\\n            for (sub, band) in sorted(self._src_per,\\n                                      key=lambda k: (str(k[0]), k[1])):\\n                if band in tuple(sbands):\\n                    self._srcp_keys.append((sub, band))\\n                    self.view_names_.append(\\n                        f\\"srcS{sub}_{band[0]:g}-{band[1]:g}\\")\\n        self._off_csp = None\\n        if cfg.use_csp:\\n            self._off_csp = len(self.view_names_)\\n            self.view_names_ += [f\\"csp_{b[0]:g}-{b[1]:g}\\" for b in cfg.bands]\\n\\n        # ---- out-of-fold meta features over the FULL training set ----- #\\n        width = 1 if n_cls == 2 else n_cls\\n        n_tr = len(train_idx)\\n        Zoof = np.zeros((n_tr, len(self.view_names_) * width))\\n\\n        skf = StratifiedKFold(n_splits=min(cfg.inner_folds, np.bincount(\\n            y_tr - y_tr.min()).min()), shuffle=True, random_state=self.seed)\\n        n_b = len(cfg.bands)\\n        for tr_in, tr_out in skf.split(np.zeros(n_tr), y_tr):\\n            g_in = train_idx[tr_in]\\n            g_out = train_idx[tr_out]\\n            for j, band in enumerate(cfg.bands):\\n                # One alignment per (fold, band), reused by every view family.\\n                # The reference is estimated on g_in only, so nothing in the\\n                # meta-feature for a held-out trial depends on that trial.\\n                ref, Call, Zall = self._cached(tb, band, g_in)\\n                Ci, Co = Call[g_in], Call[g_out]\\n                Zi, Zo = Zall[g_in], Zall[g_out]\\n\\n                m = _fit_lr(Zi, tb.y[g_in], cfg.C_self, self.seed, cfg.max_iter)\\n                c0 = j * width\\n                Zoof[np.ix_(tr_out, range(c0, c0 + width))] = \\\\\\n                    _logodds(m.predict_proba(Zo))\\n\\n                if self._has_src:\\n                    # Source models never see target labels at all, so their\\n                    # predictions on training trials are already out-of-fold.\\n                    c1 = (self._off_src + j) * width\\n                    Zoof[np.ix_(tr_out, range(c1, c1 + width))] = \\\\\\n                        _logodds(self._src_models[band].predict_proba(Zo))\\n\\n                if self._has_srcp:\\n                    for v, (sub, bd) in enumerate(self._srcp_keys):\\n                        if bd != band:\\n                            continue\\n                        c3 = (self._off_srcp + v) * width\\n                        Zoof[np.ix_(tr_out, range(c3, c3 + width))] = \\\\\\n                            _logodds(self._src_per[(sub, bd)].predict_proba(Zo))\\n\\n                if cfg.use_csp:\\n                    W = csp_from_covs(Ci, tb.y[g_in], cfg.n_csp)\\n                    lda = LDA(solver=\\"lsqr\\", shrinkage=\\"auto\\").fit(\\n                        csp_logvar_from_covs(W, Ci), tb.y[g_in])\\n                    c2 = (self._off_csp + j) * width\\n                    Zoof[np.ix_(tr_out, range(c2, c2 + width))] = _logodds(\\n                        lda.predict_proba(csp_logvar_from_covs(W, Co)))\\n\\n        # ---- refit every view on the full training set ---------------- #\\n        self._self_models, self._ref, self._csp = {}, {}, {}\\n        self._full = {}\\n        for band in cfg.bands:\\n            ref, Call, Zall = self._cached(tb, band, train_idx)\\n            Ctr = Call[train_idx]\\n            self._ref[band] = ref\\n            self._full[band] = (Call, Zall)\\n            self._self_models[band] = _fit_lr(Zall[train_idx], y_tr,\\n                                              cfg.C_self, self.seed,\\n                                              cfg.max_iter)\\n            if cfg.use_csp:\\n                W = csp_from_covs(Ctr, y_tr, cfg.n_csp)\\n                lda = LDA(solver=\\"lsqr\\", shrinkage=\\"auto\\").fit(\\n                    csp_logvar_from_covs(W, Ctr), y_tr)\\n                self._csp[band] = (W, lda)\\n\\n        # ---- fusion --------------------------------------------------- #\\n        if cfg.fusion == \\"stack\\":\\n            self._meta = _fit_lr(Zoof, y_tr, cfg.C_meta,\\n                                 self.seed, cfg.max_iter)\\n            self._uniform = False\\n        elif cfg.fusion == \\"mean\\":\\n            self._uniform = True\\n        elif cfg.fusion == \\"best\\":\\n            # Pick the single best view by OOF accuracy -- the \\"single-best\\n            # member\\" control, but selected on n_train rather than 30 trials.\\n            accs = []\\n            for v in range(len(self.view_names_)):\\n                sl = slice(v * width, (v + 1) * width)\\n                pred = self._view_argmax(Zoof[:, sl])\\n                accs.append((pred == y_tr).mean())\\n            self._best_view = int(np.argmax(accs))\\n            self._uniform = True\\n        else:\\n            raise ValueError(cfg.fusion)\\n        self._width = width\\n        return self\\n\\n    def _view_argmax(self, Z):\\n        if Z.shape[1] == 1:\\n            return np.where(Z[:, 0] > 0, self.classes_[1], self.classes_[0])\\n        return self.classes_[np.argmax(Z, axis=1)]\\n\\n    # -- predict --------------------------------------------------------- #\\n    def _meta_features(self, tb: SubjectBands, idx: np.ndarray):\\n        cfg = self.cfg\\n        width = self._width\\n        Z = np.zeros((len(idx), len(self.view_names_) * width))\\n        for j, band in enumerate(cfg.bands):\\n            Call, Zall = self._full[band]\\n            C, Zt = Call[idx], Zall[idx]\\n            c0 = j * width\\n            Z[:, c0:c0 + width] = _logodds(\\n                self._self_models[band].predict_proba(Zt))\\n            if self._has_src:\\n                c1 = (self._off_src + j) * width\\n                Z[:, c1:c1 + width] = _logodds(\\n                    self._src_models[band].predict_proba(Zt))\\n            if self._has_srcp:\\n                for v, (sub, bd) in enumerate(self._srcp_keys):\\n                    if bd != band:\\n                        continue\\n                    c3 = (self._off_srcp + v) * width\\n                    Z[:, c3:c3 + width] = _logodds(\\n                        self._src_per[(sub, bd)].predict_proba(Zt))\\n            if cfg.use_csp:\\n                W, lda = self._csp[band]\\n                c2 = (self._off_csp + j) * width\\n                Z[:, c2:c2 + width] = _logodds(\\n                    lda.predict_proba(csp_logvar_from_covs(W, C)))\\n        return Z\\n\\n    def predict_proba(self, tb: SubjectBands, idx: np.ndarray) -> np.ndarray:\\n        Z = self._meta_features(tb, idx)\\n        if self.cfg.fusion == \\"stack\\":\\n            return self._meta.predict_proba(Z)\\n        w = self._width\\n        n_v = len(self.view_names_)\\n        if self.cfg.fusion == \\"best\\":\\n            sl = slice(self._best_view * w, (self._best_view + 1) * w)\\n            L = Z[:, sl]\\n        else:\\n            L = np.mean([Z[:, v * w:(v + 1) * w] for v in range(n_v)], axis=0)\\n        if w == 1:\\n            p1 = 1.0 / (1.0 + np.exp(-L[:, 0]))\\n            return np.column_stack([1 - p1, p1])\\n        e = np.exp(L - L.max(axis=1, keepdims=True))\\n        return e / e.sum(axis=1, keepdims=True)\\n\\n    def predict(self, tb: SubjectBands, idx: np.ndarray) -> np.ndarray:\\n        return self.classes_[np.argmax(self.predict_proba(tb, idx), axis=1)]\\n"')
with open(f'{CODE_DIR}/pipeline.py', 'w') as f:
    f.write(_src)
print('wrote pipeline.py', len(_src), 'chars')


In [ ]:
import json
_src = json.loads('"\\"\\"\\"\\nbaselines_v3.py -- corrected baselines, all on identical splits.\\n\\nEvery baseline here is fixed relative to the repository version. The fixes are\\nlisted per method with the audit finding they address, because the point of\\nthis file is that the comparison is FAIR, and a reviewer must be able to check\\nthat claim without reading the diff.\\n\\nCommon protocol change (audit findings A1, B1): every method receives the same\\nepochs and does its OWN band-pass. In the repository, EEGNet and the Riemannian\\nbaseline were handed unfiltered broadband epochs while the CSP methods got four\\noptimised pass-bands. That single line depressed the Riemannian baseline by\\nroughly 20 accuracy points.\\n\\nCommon protocol change (audit finding B2): every method receives exactly the\\nsame train/test split, and any method that needs a validation set carves it\\nout of ITS OWN training portion. In the repository, EEGNet and Riemannian\\ntrained on 140 trials while DAG-SA, single-best and random search effectively\\nused 170.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom typing import Dict, List, Optional, Sequence, Tuple\\n\\nimport numpy as np\\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\\nfrom sklearn.feature_selection import mutual_info_classif\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.model_selection import train_test_split\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVC\\n\\nimport pipeline as P\\n\\nFBCSP_BANDS = tuple((4.0 + 4 * i, 8.0 + 4 * i) for i in range(7))   # 4-32 Hz\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B4  Riemannian tangent space  (the corrected strong classical baseline)\\n# --------------------------------------------------------------------------- #\\ndef riemannian_ts(tb: P.SubjectBands, tr, te, seed=0,\\n                  band=P.RIEMANNIAN_BASELINE_BAND, C=0.1):\\n    \\"\\"\\"\\n    8-30 Hz -> OAS covariance -> Euclidean alignment (reference from train)\\n    -> tangent space at identity -> L2 logistic regression.\\n\\n    This is the textbook motor-imagery Riemannian decoder. It is deliberately\\n    given the same alignment the proposed method uses, so that the comparison\\n    isolates the filter bank and the fusion layer rather than rewarding ARTS\\n    for a preprocessing step a baseline could trivially adopt.\\n    \\"\\"\\"\\n    covs = tb.covs[band]\\n    ref = P.euclidean_reference(covs[tr])\\n    Z = P.tangent(P.align(covs, ref))\\n    m = LogisticRegression(C=C, max_iter=500, random_state=seed,\\n                           class_weight=\\"balanced\\").fit(Z[tr], tb.y[tr])\\n    return m.predict(Z[te])\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B5  FBCSP  (Ang et al.) -- the strong classical competitor to a filter bank\\n# --------------------------------------------------------------------------- #\\ndef fbcsp(tb: P.SubjectBands, tr, te, seed=0, n_csp=4, k_select=8,\\n          bands: Sequence = FBCSP_BANDS):\\n    \\"\\"\\"\\n    Filter-bank CSP with mutual-information feature selection and an LDA head.\\n\\n    Included because it is the honest competitor: if the proposed method\'s\\n    gain comes from having a filter bank, then a filter-bank baseline must be\\n    in the table or the comparison is rigged. Feature selection is fit on\\n    train only.\\n    \\"\\"\\"\\n    avail = [b for b in bands if b in tb.covs]\\n    F = []\\n    for b in avail:\\n        C = tb.covs[b]\\n        W = P.csp_from_covs(C[tr], tb.y[tr], n_csp)\\n        F.append(P.csp_logvar_from_covs(W, C))\\n    F = np.concatenate(F, axis=1)\\n    k = min(k_select, F.shape[1])\\n    mi = mutual_info_classif(F[tr], tb.y[tr], random_state=seed)\\n    keep = np.argsort(mi)[::-1][:k]\\n    m = LDA(solver=\\"lsqr\\", shrinkage=\\"auto\\").fit(F[np.ix_(tr, keep)], tb.y[tr])\\n    return m.predict(F[np.ix_(te, keep)])\\n\\n\\n# --------------------------------------------------------------------------- #\\n# The CSP/CSSP-style pool that DAG-SA and its search baselines operate on\\n# --------------------------------------------------------------------------- #\\ndef build_pool(tb: P.SubjectBands, tr, seed=0, n_comps=(4, 6, 8)):\\n    \\"\\"\\"\\n    A pool of CSP log-variance views x classifier settings, in the spirit of\\n    the published pool but built from the SAME covariances every other method\\n    uses, so no method has a preprocessing advantage.\\n\\n    Each member is fit on train and exposes probabilities for all trials.\\n    Returns a list of (name, proba_all_trials).\\n    \\"\\"\\"\\n    heads = [\\n        (\\"lda\\", lambda: LDA(solver=\\"lsqr\\", shrinkage=\\"auto\\")),\\n        (\\"svm_lin\\", lambda: make_pipeline(\\n            StandardScaler(), SVC(kernel=\\"linear\\", C=1.0, probability=True,\\n                                  random_state=seed))),\\n        (\\"svm_rbf\\", lambda: make_pipeline(\\n            StandardScaler(), SVC(kernel=\\"rbf\\", C=1.0, gamma=\\"scale\\",\\n                                  probability=True, random_state=seed))),\\n        (\\"lr\\", lambda: make_pipeline(\\n            StandardScaler(), LogisticRegression(C=1.0, max_iter=500,\\n                                                 random_state=seed))),\\n    ]\\n    pool = []\\n    for band in tb.covs:\\n        for nc in n_comps:\\n            C = tb.covs[band]\\n            W = P.csp_from_covs(C[tr], tb.y[tr], nc)\\n            F = P.csp_logvar_from_covs(W, C)\\n            for hname, mk in heads:\\n                try:\\n                    m = mk().fit(F[tr], tb.y[tr])\\n                    pool.append((f\\"{band[0]:g}-{band[1]:g}|{nc}|{hname}\\",\\n                                 m.predict_proba(F)))\\n                except Exception:\\n                    # Recorded, not swallowed: a pool member that fails to fit\\n                    # is dropped here and the count is reported by the caller.\\n                    continue\\n    return pool\\n\\n\\n# ---- fusion operators over a committee of pool members -------------------- #\\ndef _fuse(op: str, probs: List[np.ndarray], idx) -> np.ndarray:\\n    S = np.stack([p[idx] for p in probs], axis=0)      # (m, n, k)\\n    if op == \\"SV\\":                                     # soft / sum rule\\n        return S.mean(axis=0)\\n    if op == \\"MIN\\":\\n        return S.min(axis=0)\\n    if op == \\"MAX\\":\\n        return S.max(axis=0)\\n    if op == \\"PROD\\":\\n        return np.exp(np.log(np.clip(S, 1e-9, 1)).mean(axis=0))\\n    if op == \\"MV\\":                                     # majority vote,\\n        hard = S.argmax(axis=2)                        # ties -> summed proba\\n        k = S.shape[2]\\n        counts = np.stack([(hard == c).sum(axis=0) for c in range(k)], axis=1)\\n        out = counts.astype(float)\\n        return out + 1e-6 * S.mean(axis=0)\\n    raise ValueError(op)\\n\\n\\nOPS = (\\"SV\\", \\"MV\\", \\"MIN\\", \\"MAX\\", \\"PROD\\")\\n\\n\\ndef _committee_acc(pool, members, op, idx, y):\\n    probs = [pool[i][1] for i in members]\\n    return (_fuse(op, probs, idx).argmax(axis=1) == y[idx]).mean()\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B1  single best pool member (selected on an inner validation split)\\n# --------------------------------------------------------------------------- #\\ndef single_best(tb, tr, te, pool, seed=0, val_frac=0.2):\\n    tr_in, tr_va = _inner_split(tb.y, tr, seed, val_frac)\\n    accs = [(p[tr_va].argmax(axis=1) == tb.y[tr_va]).mean() for _, p in pool]\\n    best = int(np.argmax(accs))\\n    return pool[best][1][te].argmax(axis=1)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B2  budgeted random search over the same ensemble space\\n# --------------------------------------------------------------------------- #\\ndef random_search(tb, tr, te, pool, seed=0, iters=300, members=4,\\n                  val_frac=0.2):\\n    rng = np.random.default_rng(seed)\\n    tr_in, tr_va = _inner_split(tb.y, tr, seed, val_frac)\\n    best, best_acc = None, -1.0\\n    n = len(pool)\\n    for _ in range(iters):\\n        mem = rng.choice(n, size=min(members, n), replace=False)\\n        op = OPS[rng.integers(len(OPS))]\\n        a = _committee_acc(pool, mem, op, tr_va, tb.y)\\n        if a > best_acc:\\n            best, best_acc = (mem, op), a\\n    mem, op = best\\n    return _fuse(op, [pool[i][1] for i in mem], te).argmax(axis=1)\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B3  DAG-SA -- the published method, with the audit\'s search bugs fixed\\n# --------------------------------------------------------------------------- #\\ndef dag_sa(tb, tr, te, pool, seed=0, iters=300, members=4, val_frac=0.2,\\n           temp0=0.05, cooling=0.98):\\n    \\"\\"\\"\\n    Simulated annealing over (member subset, fusion operator), scored on an\\n    inner validation split -- the published formulation.\\n\\n    Three audit findings are fixed so that this is the published METHOD rather\\n    than the published BUGS:\\n      * A2: the reheat no longer resets the cooling schedule, and the initial\\n        temperature (0.05) is on the scale of the objective (accuracy deltas of\\n        0.01-0.05) rather than 100x above it. The repository\'s schedule never\\n        left the near-random-acceptance regime, which made DAG-SA equivalent to\\n        random search by construction.\\n      * B6: member exclusion now compares indices, so a perturbation cannot\\n        silently duplicate a committee member.\\n      * A1 (stacking): the stacking operator is dropped rather than scored\\n        in-sample on the same validation split it is selected on. Keeping it\\n        would hand DAG-SA a 2-4 point optimistic bias on its own objective.\\n\\n    The result is a STRONGER DAG-SA than the published one. That is deliberate:\\n    the comparison should not be won by exploiting the baseline\'s defects.\\n    \\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    tr_in, tr_va = _inner_split(tb.y, tr, seed, val_frac)\\n    n = len(pool)\\n    m = min(members, n)\\n    cur = list(rng.choice(n, size=m, replace=False))\\n    cur_op = OPS[rng.integers(len(OPS))]\\n    cur_acc = _committee_acc(pool, cur, cur_op, tr_va, tb.y)\\n    best, best_op, best_acc = list(cur), cur_op, cur_acc\\n    temp = temp0\\n    for _ in range(iters):\\n        cand, cand_op = list(cur), cur_op\\n        if rng.random() < 0.5 or n <= m:\\n            cand_op = OPS[rng.integers(len(OPS))]\\n        else:\\n            j = int(rng.integers(len(cand)))\\n            choices = [i for i in range(n) if i not in cand]\\n            cand[j] = int(rng.choice(choices))\\n        a = _committee_acc(pool, cand, cand_op, tr_va, tb.y)\\n        d = a - cur_acc\\n        if d >= 0 or rng.random() < np.exp(d / max(temp, 1e-9)):\\n            cur, cur_op, cur_acc = cand, cand_op, a\\n            if a > best_acc:\\n                best, best_op, best_acc = list(cand), cand_op, a\\n        temp *= cooling\\n    return _fuse(best_op, [pool[i][1] for i in best], te).argmax(axis=1)\\n\\n\\n# --------------------------------------------------------------------------- #\\ndef _inner_split(y, tr, seed, val_frac):\\n    \\"\\"\\"Validation carved from the method\'s OWN training portion (finding B2).\\"\\"\\"\\n    a, b = train_test_split(np.arange(len(tr)), test_size=val_frac,\\n                            random_state=seed, stratify=y[tr])\\n    return tr[np.sort(a)], tr[np.sort(b)]\\n\\n\\n# --------------------------------------------------------------------------- #\\n# B6  EEGNet  (Lawhern et al. 2018) -- properly regularised and early-stopped\\n# --------------------------------------------------------------------------- #\\ndef eegnet_available() -> bool:\\n    try:\\n        import torch  # noqa: F401\\n        return True\\n    except Exception:\\n        return False\\n\\n\\ndef eegnet(X, y, tr, te, fs, seed=0, epochs=300, patience=40, lr=1e-3,\\n           batch_size=32, dropout=0.25, band=(4.0, 38.0)):\\n    \\"\\"\\"\\n    Fixes relative to the repository version (audit finding B1):\\n      * input is band-passed 4-38 Hz instead of raw broadband;\\n      * max-norm constraints (1.0 depthwise, 0.25 dense) -- the paper\'s main\\n        regulariser for small-n within-subject data -- are applied;\\n      * dropout 0.25 (within-subject) rather than 0.5 (cross-subject);\\n      * early stopping on a validation split carved from train, instead of a\\n        fixed 100 epochs with the final-epoch weights used verbatim.\\n    \\"\\"\\"\\n    import torch\\n    import torch.nn as nn\\n\\n    torch.manual_seed(seed)\\n    np.random.seed(seed)\\n    dev = torch.device(\\"cuda\\" if torch.cuda.is_available() else \\"cpu\\")\\n\\n    Xb = P.bandpass(X, band[0], band[1], fs).astype(np.float32)\\n    tr_in, tr_va = _inner_split(y, tr, seed, 0.2)\\n    mu = Xb[tr_in].mean(axis=(0, 2), keepdims=True)\\n    sd = Xb[tr_in].std(axis=(0, 2), keepdims=True) + 1e-7\\n    Xn = ((Xb - mu) / sd)[:, None, :, :]\\n\\n    n_ch, n_t = X.shape[1], X.shape[2]\\n    classes = np.unique(y)\\n    kern = max(2, fs // 2)\\n    F1, Dm = 8, 2\\n    F2 = F1 * Dm\\n\\n    class Net(nn.Module):\\n        def __init__(self):\\n            super().__init__()\\n            self.c1 = nn.Conv2d(1, F1, (1, kern), padding=(0, kern // 2),\\n                                bias=False)\\n            self.b1 = nn.BatchNorm2d(F1)\\n            self.dw = nn.Conv2d(F1, F1 * Dm, (n_ch, 1), groups=F1, bias=False)\\n            self.b2 = nn.BatchNorm2d(F1 * Dm)\\n            self.p1 = nn.AvgPool2d((1, 4))\\n            self.d1 = nn.Dropout(dropout)\\n            self.sp = nn.Conv2d(F1 * Dm, F1 * Dm, (1, 16), padding=(0, 8),\\n                                groups=F1 * Dm, bias=False)\\n            self.pw = nn.Conv2d(F1 * Dm, F2, (1, 1), bias=False)\\n            self.b3 = nn.BatchNorm2d(F2)\\n            self.p2 = nn.AvgPool2d((1, 8))\\n            self.d2 = nn.Dropout(dropout)\\n            self.fc = nn.Linear(F2 * (n_t // 32), len(classes))\\n\\n        def forward(self, x):\\n            x = self.b1(self.c1(x))\\n            x = self.d1(self.p1(nn.functional.elu(self.b2(self.dw(x)))))\\n            x = self.pw(self.sp(x))\\n            x = self.d2(self.p2(nn.functional.elu(self.b3(x))))\\n            return self.fc(x.flatten(1))\\n\\n    net = Net().to(dev)\\n    opt = torch.optim.Adam(net.parameters(), lr=lr)\\n    lossf = nn.CrossEntropyLoss()\\n    ymap = {c: i for i, c in enumerate(classes)}\\n    yy = np.array([ymap[v] for v in y])\\n\\n    def T(a, dt=torch.float32):\\n        return torch.tensor(a, dtype=dt, device=dev)\\n\\n    Xtr, Ytr = T(Xn[tr_in]), T(yy[tr_in], torch.long)\\n    Xva, Yva = T(Xn[tr_va]), T(yy[tr_va], torch.long)\\n    Xte = T(Xn[te])\\n\\n    best_state, best_va, bad = None, -1.0, 0\\n    for ep in range(epochs):\\n        net.train()\\n        perm = torch.randperm(len(Xtr), device=dev)\\n        for i in range(0, len(perm), batch_size):\\n            b = perm[i:i + batch_size]\\n            opt.zero_grad()\\n            lossf(net(Xtr[b]), Ytr[b]).backward()\\n            opt.step()\\n            with torch.no_grad():           # max-norm constraints\\n                w = net.dw.weight\\n                w.copy_(_maxnorm(w, 1.0, dims=(1, 2, 3)))\\n                w = net.fc.weight\\n                w.copy_(_maxnorm(w, 0.25, dims=(1,)))\\n        net.eval()\\n        with torch.no_grad():\\n            va = (net(Xva).argmax(1) == Yva).float().mean().item()\\n        if va > best_va:\\n            best_va, bad = va, 0\\n            best_state = {k: v.detach().clone()\\n                          for k, v in net.state_dict().items()}\\n        else:\\n            bad += 1\\n            if bad >= patience:\\n                break\\n    if best_state is not None:\\n        net.load_state_dict(best_state)\\n    net.eval()\\n    with torch.no_grad():\\n        pred = net(Xte).argmax(1).cpu().numpy()\\n    return classes[pred]\\n\\n\\ndef _maxnorm(w, m, dims):\\n    import torch\\n    n = w.norm(2, dim=dims, keepdim=True).clamp(min=1e-12)\\n    return w * (n.clamp(max=m) / n)\\n"')
with open(f'{CODE_DIR}/baselines_v3.py', 'w') as f:
    f.write(_src)
print('wrote baselines_v3.py', len(_src), 'chars')


In [ ]:
import json
_src = json.loads('"\\"\\"\\"\\nprotocol.py -- the pre-registered confirmatory evaluation.\\n\\nThe pre-registration below is frozen BEFORE the confirmatory run. Everything\\nthat could be chosen to flatter the proposed method -- datasets, subjects,\\nseeds, split fractions, baselines, metric, unit of analysis, statistical test,\\nmultiplicity correction, and the success criterion -- is fixed here in code and\\nwritten verbatim to `preregistration.json` at the start of the run.\\n\\nThe run is executed once. Results are appended per unit and never regenerated.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport os\\nimport sys\\nimport time\\nimport traceback\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nfrom typing import Dict, List, Optional, Sequence\\n\\nimport numpy as np\\n\\nimport baselines_v3 as B\\nimport data as D\\nimport pipeline as P\\n\\n# =========================================================================== #\\n#                            PRE-REGISTRATION\\n# =========================================================================== #\\nPREREG = {\\n    \\"title\\": \\"Confirmatory evaluation of ARTS against corrected baselines\\",\\n    \\"frozen\\": \\"2026-07-29\\",\\n\\n    \\"datasets\\": {\\n        \\"ds1\\": {\\n            \\"subjects\\": D.DS1_ALL,\\n            \\"note\\": \\"BCI IV Dataset 1. Subjects c,d,e are ARTIFICIALLY \\"\\n                    \\"GENERATED by the competition organisers and are reported \\"\\n                    \\"separately in the per-subject table. The primary ds1 \\"\\n                    \\"analysis is pre-specified on ALL SEVEN; a secondary \\"\\n                    \\"analysis on the four real subjects (a,b,f,g) is also \\"\\n                    \\"pre-specified because the artificial subjects are near \\"\\n                    \\"ceiling and could mask a difference.\\",\\n            \\"classes\\": \\"two per subject, but NOT the same two for every \\"\\n                       \\"subject (a,f = left/foot; b,c,d,e,g = left/right). \\"\\n                       \\"Cross-subject transfer is therefore restricted to \\"\\n                       \\"source subjects sharing the target\'s class pair.\\",\\n            \\"window_s\\": [0.5, 4.0],\\n        },\\n        \\"ds2a_binary\\": {\\n            \\"subjects\\": D.DS2A_ALL,\\n            \\"window_s\\": [2.0, 6.0],\\n            \\"note\\": \\"Left vs right hand. Window is the standard 4 s MI period \\"\\n                    \\"beginning 2 s after fixation onset. The repository used \\"\\n                    \\"(0.5,4.5), which is mostly fixation and cue.\\",\\n        },\\n        \\"ds2a_4class\\": {\\n            \\"subjects\\": D.DS2A_ALL,\\n            \\"window_s\\": [2.0, 6.0],\\n            \\"note\\": \\"Pre-specified as a secondary confirmatory analysis: it \\"\\n                    \\"has twice the trials, so it is the better-powered test.\\",\\n        },\\n    },\\n\\n    \\"seeds\\": [42, 43, 44, 45, 46, 47, 48, 49, 50, 51],\\n\\n    \\"split\\": {\\n        \\"scheme\\": \\"stratified train/test, single split per (subject, seed)\\",\\n        \\"test_fraction\\": 0.30,\\n        \\"note\\": \\"Raised from the repository\'s 0.15. With 22-30 test trials an \\"\\n                \\"exact McNemar test essentially cannot reach p<0.05, so the \\"\\n                \\"repository\'s 95% tie rate was partly mechanical (audit B3).\\",\\n        \\"validation\\": \\"Methods requiring model selection carve 20% of their \\"\\n                      \\"OWN training portion. No method sees more data than \\"\\n                      \\"another (audit B2).\\",\\n    },\\n\\n    \\"methods\\": {\\n        \\"ARTS\\": \\"proposed: 6-band Riemannian tangent-space views, Euclidean \\"\\n                \\"alignment, per-source-subject transfer views on 8-30 Hz, \\"\\n                \\"out-of-fold stacked logistic fusion\\",\\n        \\"riemannian_ts\\": \\"B4 corrected: 8-30 Hz, OAS, EA, tangent space, L2 LR\\",\\n        \\"fbcsp\\": \\"B5: filter-bank CSP + mutual-information selection + LDA\\",\\n        \\"eegnet\\": \\"B6 corrected: 4-38 Hz input, max-norm, early stopping\\",\\n        \\"single_best\\": \\"B1: best single CSP pool member, selected on inner val\\",\\n        \\"random_search\\": \\"B2: 300-draw budgeted random search over the same \\"\\n                         \\"ensemble space, selected on inner val\\",\\n        \\"dag_sa\\": \\"B3: the published method with its search bugs fixed\\",\\n    },\\n\\n    \\"primary_metric\\": \\"accuracy\\",\\n    \\"secondary_metrics\\": [\\"cohen_kappa\\", \\"balanced_accuracy\\",\\n                          \\"per_class_precision_recall_f1\\"],\\n\\n    \\"unit_of_analysis\\": {\\n        \\"primary\\": \\"subject\\",\\n        \\"rationale\\": \\"Seeds are re-splits of the same trials, so seed-level \\"\\n                     \\"replicates are not independent and a CI over seeds \\"\\n                     \\"measures split noise, not generalisation to a new \\"\\n                     \\"subject (audit A4). Accuracy is averaged over the 10 \\"\\n                     \\"seeds within a subject first; the CI is then taken \\"\\n                     \\"across subjects with a t critical value on n-1 df.\\",\\n        \\"secondary\\": \\"(subject, seed) unit, for the paired McNemar analysis\\",\\n    },\\n\\n    \\"statistical_test\\": {\\n        \\"test\\": \\"exact two-sided McNemar, paired on identical test trials\\",\\n        \\"applied_at\\": \\"each (subject, seed) unit\\",\\n        \\"multiplicity\\": \\"Holm-Bonferroni within each (proposed vs baseline) \\"\\n                        \\"comparison across its units\\",\\n        \\"gate\\": \\"a unit counts as a win/loss only if its Holm-corrected \\"\\n                \\"p < 0.05; otherwise it is a tie\\",\\n        \\"paired_ci\\": \\"paired t interval on the per-subject differences\\",\\n    },\\n\\n    \\"success_criterion\\": {\\n        \\"primary\\": \\"On a given dataset, ARTS is declared superior only if, \\"\\n                   \\"against EVERY baseline, the lower bound of the 95% paired \\"\\n                   \\"CI on the per-subject accuracy difference is > 0 AND the \\"\\n                   \\"point estimate is >= 2.0 accuracy points.\\",\\n        \\"equivalence\\": \\"A difference whose 95% CI lies entirely within \\"\\n                       \\"+/- 2.0 points is reported as a TIE, not a win.\\",\\n        \\"indeterminate\\": \\"Anything else is reported as inconclusive at this \\"\\n                         \\"sample size, with the CI given.\\",\\n        \\"no_post_hoc\\": \\"No baseline, subject, seed or metric may be dropped \\"\\n                       \\"after seeing results. The table is reported in full.\\",\\n    },\\n\\n    \\"development_disclosure\\": [\\n        \\"Method design used ONLY cross-validation inside the training \\"\\n        \\"portion, at development seeds 900-902, which are disjoint from the \\"\\n        \\"confirmatory seeds 42-51.\\",\\n        \\"Two exceptions are disclosed: (a) a label-permutation leakage check \\"\\n        \\"on ds1 subject a scored held-out folds at seeds 0-4; (b) \\"\\n        \\"`reference_check.py` scored 5-fold CV for textbook CSP+LDA and \\"\\n        \\"tangent-space baselines on both datasets at seed 0. Both were \\"\\n        \\"diagnostics on baselines, not tuning of ARTS, but they did inform \\"\\n        \\"the judgement that the manuscript\'s accuracy LEVEL is a \\"\\n        \\"preprocessing artefact.\\",\\n        \\"Frozen ARTS hyperparameters: 6 bands (4-8, 8-12, 12-16, 16-22, \\"\\n        \\"22-30, 8-30 Hz); C_self=C_src=0.1; C_meta=1.0; 5 inner folds; \\"\\n        \\"per-source-subject transfer views on the 8-30 Hz band only.\\",\\n    ],\\n}\\n\\n\\n# =========================================================================== #\\n#                                 RUNNER\\n# =========================================================================== #\\n@dataclass\\nclass RunCfg:\\n    ds1_dir: str = \\"\\"\\n    ds2a_dir: str = \\"\\"\\n    out_dir: str = \\"results_v3\\"\\n    datasets: Sequence[str] = (\\"ds1\\", \\"ds2a_binary\\", \\"ds2a_4class\\")\\n    seeds: Sequence[int] = tuple(PREREG[\\"seeds\\"])\\n    test_fraction: float = 0.30\\n    run_eegnet: bool = True\\n    run_ablation: bool = True\\n\\n\\nARTS_FROZEN = P.ARTSConfig(\\n    bands=P.DEFAULT_BANDS,\\n    use_alignment=True,\\n    use_transfer=True,\\n    src_mode=\\"per_subject\\",\\n    src_bands=(P.RIEMANNIAN_BASELINE_BAND,),\\n    use_csp=False,\\n    fusion=\\"stack\\",\\n    C_self=0.1, C_src=0.1, C_meta=1.0,\\n)\\n\\n# The ablation ladder. Each row differs from the row above by ONE component,\\n# so the attribution of any gain is unambiguous.\\nABLATION = {\\n    \\"A0_single_band\\":  P.ARTSConfig(bands=(P.RIEMANNIAN_BASELINE_BAND,),\\n                                    use_transfer=False, fusion=\\"mean\\"),\\n    \\"A1_filterbank_mean\\": P.ARTSConfig(use_transfer=False, fusion=\\"mean\\"),\\n    \\"A2_filterbank_stack\\": P.ARTSConfig(use_transfer=False, fusion=\\"stack\\"),\\n    \\"A3_no_alignment\\": P.ARTSConfig(use_transfer=False, fusion=\\"stack\\",\\n                                    use_alignment=False),\\n    \\"A4_transfer_pooled\\": P.ARTSConfig(src_mode=\\"pooled\\"),\\n    \\"A5_full\\": ARTS_FROZEN,\\n}\\n\\n\\ndef setup_logging(out_dir: Path):\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    log = logging.getLogger(\\"v3\\")\\n    log.setLevel(logging.INFO)\\n    log.handlers.clear()\\n    fh = logging.FileHandler(out_dir / \\"run.log\\")\\n    sh = logging.StreamHandler(sys.stdout)\\n    fmt = logging.Formatter(\\"%(asctime)s %(levelname)s %(message)s\\")\\n    for h in (fh, sh):\\n        h.setFormatter(fmt)\\n        log.addHandler(h)\\n    return log\\n\\n\\ndef _load_all(dataset: str, cfg: RunCfg, log):\\n    \\"\\"\\"\\n    Load every subject of a dataset and precompute band covariances.\\n\\n    Covariances are cached to disk. They depend only on the raw recording, the\\n    epoch window and the band definitions -- never on a split, a seed or a\\n    label -- so caching them cannot leak anything, and it makes a resumed run\\n    start in seconds instead of minutes.\\n    \\"\\"\\"\\n    d = cfg.ds1_dir if dataset == \\"ds1\\" else cfg.ds2a_dir\\n    cache_dir = Path(cfg.out_dir) / \\"cov_cache\\"\\n    cache_dir.mkdir(parents=True, exist_ok=True)\\n    units, raw = {}, {}\\n    for s in D.subjects_for(dataset):\\n        u = D.load_unit(dataset, d, s)\\n        raw[s] = u\\n        cp = cache_dir / f\\"{dataset}_{s}.npz\\"\\n        bands = list(P.DEFAULT_BANDS) + [b for b in B.FBCSP_BANDS\\n                                         if b not in P.DEFAULT_BANDS]\\n        if cp.exists():\\n            z = np.load(cp, allow_pickle=False)\\n            covs = {bands[i]: z[f\\"b{i}\\"] for i in range(len(bands))}\\n            units[s] = P.SubjectBands(covs=covs, y=z[\\"y\\"], subject=s)\\n        else:\\n            units[s] = P.SubjectBands.build(u[\\"X\\"], u[\\"y\\"], u[\\"fs\\"], s,\\n                                            bands=bands)\\n            np.savez_compressed(\\n                cp, y=units[s].y,\\n                **{f\\"b{i}\\": units[s].covs[b] for i, b in enumerate(bands)})\\n        log.info(f\\"  loaded {dataset} subject {s}: X={u[\'X\'].shape} \\"\\n                 f\\"classes={u[\'class_names\']}\\")\\n    return units, raw\\n\\n\\ndef _sources_for(dataset, s, units, raw):\\n    \\"\\"\\"\\n    Source subjects for transfer.\\n\\n    Dataset 1\'s class pair differs by subject, so a source subject is only\\n    usable if its two classes are the SAME two classes as the target\'s.\\n    Ignoring this asks a source model to map \'foot\' and \'right hand\' onto the\\n    same output, which is why pooled transfer was worthless on ds1 in\\n    development.\\n    \\"\\"\\"\\n    out = []\\n    for o in units:\\n        if o == s:\\n            continue\\n        if dataset == \\"ds1\\" and raw[o][\\"class_names\\"] != raw[s][\\"class_names\\"]:\\n            continue\\n        out.append(units[o])\\n    return out\\n\\n\\ndef run(cfg: RunCfg):\\n    out = Path(cfg.out_dir)\\n    log = setup_logging(out)\\n    (out / \\"preregistration.json\\").write_text(json.dumps(PREREG, indent=2))\\n    (out / \\"run_config.json\\").write_text(json.dumps(asdict(cfg), indent=2))\\n    log.info(\\"pre-registration written; beginning confirmatory run\\")\\n\\n    pred_path = out / \\"predictions.jsonl\\"\\n    done = set()\\n    if pred_path.exists():\\n        for line in pred_path.read_text().splitlines():\\n            if line.strip():\\n                r = json.loads(line)\\n                done.add((r[\\"dataset\\"], str(r[\\"subject\\"]), r[\\"seed\\"]))\\n        log.info(f\\"resuming: {len(done)} units already complete\\")\\n\\n    have_torch = cfg.run_eegnet and B.eegnet_available()\\n    if cfg.run_eegnet and not have_torch:\\n        log.warning(\\"torch unavailable -- EEGNet will be recorded as SKIPPED, \\"\\n                    \\"not silently omitted\\")\\n\\n    for dataset in cfg.datasets:\\n        log.info(f\\"=== dataset {dataset} ===\\")\\n        units, raw = _load_all(dataset, cfg, log)\\n        # Source models depend only on other subjects\' data, never on the\\n        # target\'s split, so they are fitted once per target and reused for\\n        # every seed and every ablation row that shares their configuration.\\n        src_cache: Dict = {}\\n\\n        for s in D.subjects_for(dataset):\\n            tb = units[s]\\n            srcs = _sources_for(dataset, s, units, raw)\\n            for seed in cfg.seeds:\\n                key = (dataset, str(s), seed)\\n                if key in done:\\n                    continue\\n                t0 = time.time()\\n                try:\\n                    rec = _run_unit(dataset, s, seed, tb, srcs, raw[s], cfg,\\n                                    src_cache, have_torch, log)\\n                except Exception as e:\\n                    log.error(f\\"unit {key} FAILED: {e}\\\\n{traceback.format_exc()}\\")\\n                    continue\\n                rec[\\"secs\\"] = round(time.time() - t0, 1)\\n                with open(pred_path, \\"a\\") as f:\\n                    f.write(json.dumps(rec) + \\"\\\\n\\")\\n                    f.flush()\\n                    os.fsync(f.fileno())\\n                log.info(f\\"  {dataset} s={s} seed={seed} done in \\"\\n                         f\\"{rec[\'secs\']}s :: \\" +\\n                         \\"  \\".join(f\\"{k}={np.mean(np.array(v)==np.array(rec[\'y_true\'])):.3f}\\"\\n                                   for k, v in rec[\\"pred\\"].items()\\n                                   if v is not None))\\n        del units, raw, src_cache\\n    log.info(\\"run complete\\")\\n\\n\\ndef _run_unit(dataset, s, seed, tb, srcs, rawu, cfg, src_cache, have_torch,\\n              log):\\n    from sklearn.model_selection import train_test_split\\n    y = tb.y\\n    idx = np.arange(len(y))\\n    tr, te = train_test_split(idx, test_size=cfg.test_fraction,\\n                              random_state=seed, stratify=y)\\n    tr, te = np.sort(tr), np.sort(te)\\n\\n    rec = {\\"dataset\\": dataset, \\"subject\\": s, \\"seed\\": seed,\\n           \\"y_true\\": y[te].tolist(), \\"test_idx\\": te.tolist(),\\n           \\"class_names\\": rawu[\\"class_names\\"], \\"pred\\": {}, \\"notes\\": {}}\\n\\n    # ---- proposed method + ablation ladder --------------------------- #\\n    rows = {\\"ARTS\\": ARTS_FROZEN}\\n    if cfg.run_ablation:\\n        rows.update(ABLATION)\\n    feat_cache: Dict = {}      # shared across configs, dropped after the unit\\n    for name, acfg in rows.items():\\n        m = P.ARTS(acfg, seed=seed, feature_cache=feat_cache)\\n        if acfg.use_transfer and srcs:\\n            ck = (str(s), acfg.src_mode, tuple(acfg.src_bands or acfg.bands),\\n                  acfg.use_alignment, acfg.C_src)\\n            if ck not in src_cache:\\n                m0 = P.ARTS(acfg, seed=seed).fit_sources(srcs)\\n                if acfg.src_mode == \\"per_subject\\":\\n                    m0.fit_sources_per_subject(srcs, acfg.src_bands)\\n                src_cache[ck] = (m0._src_models, getattr(m0, \\"_src_per\\", {}))\\n            m._src_models, m._src_per = src_cache[ck]\\n        elif acfg.use_transfer and not srcs:\\n            # ds1 subjects whose class pair is unique have no usable source.\\n            rec[\\"notes\\"][name] = \\"no class-matched source subjects\\"\\n            m._src_models = {}\\n        m.fit(tb, tr, srcs)\\n        rec[\\"pred\\"][name] = m.predict(tb, te).tolist()\\n        if name == \\"ARTS\\":\\n            rec[\\"notes\\"][\\"n_views\\"] = len(m.view_names_)\\n            rec[\\"notes\\"][\\"view_names\\"] = m.view_names_\\n\\n    # ---- baselines --------------------------------------------------- #\\n    rec[\\"pred\\"][\\"riemannian_ts\\"] = B.riemannian_ts(tb, tr, te, seed).tolist()\\n    rec[\\"pred\\"][\\"fbcsp\\"] = B.fbcsp(tb, tr, te, seed).tolist()\\n\\n    pool = B.build_pool(tb, tr, seed)\\n    rec[\\"notes\\"][\\"pool_size\\"] = len(pool)\\n    rec[\\"pred\\"][\\"single_best\\"] = B.single_best(tb, tr, te, pool, seed).tolist()\\n    rec[\\"pred\\"][\\"random_search\\"] = B.random_search(tb, tr, te, pool,\\n                                                   seed).tolist()\\n    rec[\\"pred\\"][\\"dag_sa\\"] = B.dag_sa(tb, tr, te, pool, seed).tolist()\\n\\n    if have_torch:\\n        try:\\n            rec[\\"pred\\"][\\"eegnet\\"] = B.eegnet(\\n                rawu[\\"X\\"], y, tr, te, rawu[\\"fs\\"], seed).tolist()\\n        except Exception as e:\\n            rec[\\"pred\\"][\\"eegnet\\"] = None\\n            rec[\\"notes\\"][\\"eegnet_error\\"] = str(e)\\n            log.error(f\\"EEGNet failed on {dataset} s={s} seed={seed}: {e}\\")\\n    else:\\n        rec[\\"pred\\"][\\"eegnet\\"] = None\\n        rec[\\"notes\\"][\\"eegnet\\"] = \\"SKIPPED (torch unavailable)\\"\\n    return rec\\n"')
with open(f'{CODE_DIR}/protocol.py', 'w') as f:
    f.write(_src)
print('wrote protocol.py', len(_src), 'chars')


In [ ]:
import json
_src = json.loads('"\\"\\"\\"\\nanalysis.py -- turns predictions.jsonl into the pre-registered tables.\\n\\nStatistical choices, and why they differ from the repository (audit A4, B3):\\n\\n  * The unit of analysis is the SUBJECT, not the (subject, seed) pair. Seeds\\n    are re-splits of the same trials, so seed-level replicates are strongly\\n    dependent; a CI taken over them measures split-assignment noise rather\\n    than uncertainty about a new subject. Accuracy is averaged over seeds\\n    within a subject, and the interval is taken across subjects.\\n  * The critical value is Student\'s t on n-1 df, not 1.96. With n=7 or n=9\\n    subjects, z understates the interval by 20-30%.\\n  * McNemar p-values are Holm-corrected across the units of a comparison\\n    before being counted as wins or losses. The repository counted 160\\n    uncorrected tests at alpha=0.05.\\n  * Comparisons are reported as paired differences with a paired CI, because\\n    every method is evaluated on identical splits -- the pairing is the main\\n    source of power and discarding it would be wasteful.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import defaultdict\\nfrom pathlib import Path\\nfrom typing import Dict, List, Optional, Sequence\\n\\nimport numpy as np\\nfrom scipy import stats\\nfrom sklearn.metrics import (balanced_accuracy_score, cohen_kappa_score,\\n                             confusion_matrix, precision_recall_fscore_support)\\n\\nPROPOSED = \\"ARTS\\"\\nBASELINES = [\\"riemannian_ts\\", \\"fbcsp\\", \\"eegnet\\", \\"single_best\\",\\n             \\"random_search\\", \\"dag_sa\\"]\\nABLATION_ROWS = [\\"A0_single_band\\", \\"A1_filterbank_mean\\", \\"A2_filterbank_stack\\",\\n                 \\"A3_no_alignment\\", \\"A4_transfer_pooled\\", \\"A5_full\\"]\\nEQUIV_MARGIN = 2.0          # percentage points, pre-registered\\n\\n\\n# --------------------------------------------------------------------------- #\\ndef load(path) -> List[dict]:\\n    recs = []\\n    for line in Path(path).read_text().splitlines():\\n        if line.strip():\\n            recs.append(json.loads(line))\\n    return recs\\n\\n\\ndef exact_mcnemar(a_ok: np.ndarray, b_ok: np.ndarray) -> float:\\n    \\"\\"\\"Exact two-sided McNemar on the discordant pairs.\\"\\"\\"\\n    n01 = int(np.sum(~a_ok & b_ok))\\n    n10 = int(np.sum(a_ok & ~b_ok))\\n    n = n01 + n10\\n    if n == 0:\\n        return 1.0\\n    return float(min(1.0, 2.0 * stats.binom.cdf(min(n01, n10), n, 0.5)))\\n\\n\\ndef holm(pvals: Sequence[float]) -> np.ndarray:\\n    \\"\\"\\"Holm-Bonferroni step-down adjusted p-values.\\"\\"\\"\\n    p = np.asarray(pvals, dtype=float)\\n    m = len(p)\\n    order = np.argsort(p)\\n    adj = np.empty(m)\\n    running = 0.0\\n    for i, j in enumerate(order):\\n        running = max(running, (m - i) * p[j])\\n        adj[j] = min(1.0, running)\\n    return adj\\n\\n\\ndef t_ci(x: Sequence[float], alpha=0.05):\\n    x = np.asarray(x, dtype=float)\\n    n = len(x)\\n    m = float(x.mean())\\n    if n < 2:\\n        return m, np.nan, np.nan, 0.0\\n    sd = float(x.std(ddof=1))\\n    h = stats.t.ppf(1 - alpha / 2, n - 1) * sd / np.sqrt(n)\\n    return m, m - h, m + h, sd\\n\\n\\n# --------------------------------------------------------------------------- #\\ndef per_unit_table(recs) -> List[dict]:\\n    rows = []\\n    for r in recs:\\n        yt = np.asarray(r[\\"y_true\\"])\\n        for meth, pred in r[\\"pred\\"].items():\\n            if pred is None:\\n                continue\\n            yp = np.asarray(pred)\\n            rows.append(dict(\\n                dataset=r[\\"dataset\\"], subject=str(r[\\"subject\\"]),\\n                seed=r[\\"seed\\"], method=meth,\\n                accuracy=float((yp == yt).mean()),\\n                kappa=float(cohen_kappa_score(yt, yp)),\\n                balanced=float(balanced_accuracy_score(yt, yp)),\\n            ))\\n    return rows\\n\\n\\ndef method_summary(rows, dataset, methods=None) -> List[dict]:\\n    \\"\\"\\"Per-method summary with the SUBJECT as the unit of analysis.\\"\\"\\"\\n    by = defaultdict(lambda: defaultdict(list))\\n    for r in rows:\\n        if r[\\"dataset\\"] != dataset:\\n            continue\\n        by[r[\\"method\\"]][r[\\"subject\\"]].append(r)\\n    out = []\\n    for meth, subs in by.items():\\n        if methods and meth not in methods:\\n            continue\\n        acc_s = [np.mean([x[\\"accuracy\\"] for x in v]) for v in subs.values()]\\n        kap_s = [np.mean([x[\\"kappa\\"] for x in v]) for v in subs.values()]\\n        bal_s = [np.mean([x[\\"balanced\\"] for x in v]) for v in subs.values()]\\n        m, lo, hi, sd = t_ci(acc_s)\\n        km, klo, khi, ksd = t_ci(kap_s)\\n        out.append(dict(\\n            dataset=dataset, method=meth,\\n            acc_mean=100 * m, acc_sd=100 * sd,\\n            acc_ci_low=100 * lo, acc_ci_high=100 * hi,\\n            kappa_mean=km, kappa_ci_low=klo, kappa_ci_high=khi,\\n            balanced_mean=100 * float(np.mean(bal_s)),\\n            n_subjects=len(subs),\\n            n_units=sum(len(v) for v in subs.values()),\\n        ))\\n    return sorted(out, key=lambda d: -d[\\"acc_mean\\"])\\n\\n\\ndef paired_comparison(rows, dataset, a=PROPOSED, b=\\"riemannian_ts\\") -> dict:\\n    \\"\\"\\"Paired per-subject difference a - b, with a paired t interval.\\"\\"\\"\\n    acc = defaultdict(dict)\\n    for r in rows:\\n        if r[\\"dataset\\"] != dataset:\\n            continue\\n        acc[r[\\"method\\"]].setdefault(r[\\"subject\\"], []).append(r[\\"accuracy\\"])\\n    if a not in acc or b not in acc:\\n        return {}\\n    subs = sorted(set(acc[a]) & set(acc[b]))\\n    if not subs:\\n        return {}\\n    d = [np.mean(acc[a][s]) - np.mean(acc[b][s]) for s in subs]\\n    m, lo, hi, sd = t_ci(d)\\n    tstat, p = stats.ttest_rel(\\n        [np.mean(acc[a][s]) for s in subs],\\n        [np.mean(acc[b][s]) for s in subs]) if len(subs) > 1 else (np.nan, np.nan)\\n    return dict(dataset=dataset, proposed=a, baseline=b, n_subjects=len(subs),\\n                diff_mean=100 * m, diff_ci_low=100 * lo, diff_ci_high=100 * hi,\\n                diff_sd=100 * sd, t=float(tstat), p=float(p),\\n                per_subject={s: 100 * v for s, v in zip(subs, d)})\\n\\n\\ndef mcnemar_wtl(recs, dataset, a=PROPOSED, b=\\"riemannian_ts\\") -> dict:\\n    \\"\\"\\"Significance-gated win/tie/loss over (subject, seed) units.\\"\\"\\"\\n    units, praw, sign = [], [], []\\n    for r in recs:\\n        if r[\\"dataset\\"] != dataset:\\n            continue\\n        pa, pb = r[\\"pred\\"].get(a), r[\\"pred\\"].get(b)\\n        if pa is None or pb is None:\\n            continue\\n        yt = np.asarray(r[\\"y_true\\"])\\n        a_ok = np.asarray(pa) == yt\\n        b_ok = np.asarray(pb) == yt\\n        units.append((str(r[\\"subject\\"]), r[\\"seed\\"]))\\n        praw.append(exact_mcnemar(a_ok, b_ok))\\n        sign.append(int(np.sign(a_ok.mean() - b_ok.mean())))\\n    if not units:\\n        return {}\\n    padj = holm(praw)\\n    w = int(np.sum((padj < 0.05) & (np.array(sign) > 0)))\\n    l = int(np.sum((padj < 0.05) & (np.array(sign) < 0)))\\n    return dict(dataset=dataset, proposed=a, baseline=b, n_units=len(units),\\n                wins=w, losses=l, ties=len(units) - w - l,\\n                wins_uncorrected=int(np.sum((np.array(praw) < 0.05)\\n                                            & (np.array(sign) > 0))),\\n                losses_uncorrected=int(np.sum((np.array(praw) < 0.05)\\n                                              & (np.array(sign) < 0))))\\n\\n\\ndef class_metrics(recs, dataset, method=PROPOSED) -> dict:\\n    yt, yp, names = [], [], None\\n    for r in recs:\\n        if r[\\"dataset\\"] != dataset or r[\\"pred\\"].get(method) is None:\\n            continue\\n        yt += r[\\"y_true\\"]\\n        yp += r[\\"pred\\"][method]\\n        names = r[\\"class_names\\"]\\n    if not yt:\\n        return {}\\n    labs = sorted(set(yt) | set(yp))\\n    pr, rc, f1, sup = precision_recall_fscore_support(\\n        yt, yp, labels=labs, zero_division=0)\\n    nm = [names[i] if names and i < len(names) else str(i) for i in labs]\\n    return dict(dataset=dataset, method=method,\\n                classes={n: dict(precision=float(a), recall=float(b),\\n                                 f1=float(c), support=int(d))\\n                         for n, a, b, c, d in zip(nm, pr, rc, f1, sup)},\\n                confusion=confusion_matrix(yt, yp, labels=labs).tolist(),\\n                confusion_labels=nm)\\n\\n\\ndef verdict(cmp_rows: List[dict]) -> dict:\\n    \\"\\"\\"Apply the pre-registered success criterion literally.\\"\\"\\"\\n    if not cmp_rows:\\n        return {\\"verdict\\": \\"no data\\"}\\n    worst = min(cmp_rows, key=lambda c: c[\\"diff_ci_low\\"])\\n    all_positive = all(c[\\"diff_ci_low\\"] > 0 for c in cmp_rows)\\n    all_big = all(c[\\"diff_mean\\"] >= EQUIV_MARGIN for c in cmp_rows)\\n    if all_positive and all_big:\\n        v = \\"SUPERIOR\\"\\n        why = (\\"every baseline\'s 95% paired CI lower bound is above zero and \\"\\n               f\\"every point estimate is >= {EQUIV_MARGIN} points\\")\\n    elif all(abs(c[\\"diff_ci_low\\"]) < EQUIV_MARGIN\\n             and abs(c[\\"diff_ci_high\\"]) < EQUIV_MARGIN for c in cmp_rows):\\n        v = \\"TIE (equivalent)\\"\\n        why = (f\\"every CI lies entirely within +/-{EQUIV_MARGIN} points\\")\\n    else:\\n        v = \\"INCONCLUSIVE\\"\\n        why = (\\"at least one comparison has a CI that neither excludes zero \\"\\n               \\"nor lies inside the equivalence margin\\")\\n    return dict(verdict=v, reason=why,\\n                binding_comparison=worst[\\"baseline\\"],\\n                binding_diff=worst[\\"diff_mean\\"],\\n                binding_ci=[worst[\\"diff_ci_low\\"], worst[\\"diff_ci_high\\"]])\\n\\n\\n# --------------------------------------------------------------------------- #\\ndef full_report(pred_path, out_dir=None) -> dict:\\n    recs = load(pred_path)\\n    rows = per_unit_table(recs)\\n    datasets = sorted({r[\\"dataset\\"] for r in rows})\\n    report = {\\"datasets\\": {}}\\n\\n    for ds in datasets:\\n        main = [PROPOSED] + BASELINES\\n        summ = method_summary(rows, ds, methods=set(main))\\n        abl = method_summary(rows, ds, methods=set(ABLATION_ROWS))\\n        cmps, wtl = [], []\\n        for b in BASELINES:\\n            c = paired_comparison(rows, ds, PROPOSED, b)\\n            if c:\\n                cmps.append(c)\\n                wtl.append(mcnemar_wtl(recs, ds, PROPOSED, b))\\n        report[\\"datasets\\"][ds] = dict(\\n            summary=summ, ablation=abl, comparisons=cmps,\\n            win_tie_loss=[w for w in wtl if w],\\n            class_metrics=class_metrics(recs, ds, PROPOSED),\\n            per_subject=_per_subject(rows, ds, main),\\n            verdict=verdict(cmps),\\n        )\\n\\n    # ds1 secondary analysis on the four real subjects\\n    if \\"ds1\\" in datasets:\\n        real = [r for r in rows if r[\\"dataset\\"] != \\"ds1\\"\\n                or r[\\"subject\\"] in (\\"a\\", \\"b\\", \\"f\\", \\"g\\")]\\n        c2 = [paired_comparison(real, \\"ds1\\", PROPOSED, b) for b in BASELINES]\\n        c2 = [c for c in c2 if c]\\n        report[\\"ds1_real_subjects_only\\"] = dict(\\n            summary=method_summary(real, \\"ds1\\",\\n                                   methods=set([PROPOSED] + BASELINES)),\\n            comparisons=c2, verdict=verdict(c2))\\n\\n    if out_dir:\\n        p = Path(out_dir)\\n        p.mkdir(parents=True, exist_ok=True)\\n        (p / \\"report.json\\").write_text(json.dumps(report, indent=2))\\n        (p / \\"report.md\\").write_text(render_markdown(report))\\n    return report\\n\\n\\ndef _per_subject(rows, ds, methods):\\n    out = defaultdict(dict)\\n    agg = defaultdict(lambda: defaultdict(list))\\n    for r in rows:\\n        if r[\\"dataset\\"] == ds and r[\\"method\\"] in methods:\\n            agg[r[\\"subject\\"]][r[\\"method\\"]].append(r[\\"accuracy\\"])\\n    for s, mm in agg.items():\\n        for m, v in mm.items():\\n            out[s][m] = round(100 * float(np.mean(v)), 2)\\n    return dict(out)\\n\\n\\ndef render_markdown(report: dict) -> str:\\n    L = [\\"# Confirmatory evaluation -- results\\", \\"\\"]\\n    for ds, d in report[\\"datasets\\"].items():\\n        L += [f\\"## {ds}\\", \\"\\",\\n              \\"| method | acc mean | sd | 95% CI | kappa | n subj | n units |\\",\\n              \\"|---|---|---|---|---|---|---|\\"]\\n        for r in d[\\"summary\\"]:\\n            L.append(f\\"| {r[\'method\']} | {r[\'acc_mean\']:.2f} | \\"\\n                     f\\"{r[\'acc_sd\']:.2f} | [{r[\'acc_ci_low\']:.2f}, \\"\\n                     f\\"{r[\'acc_ci_high\']:.2f}] | {r[\'kappa_mean\']:.3f} | \\"\\n                     f\\"{r[\'n_subjects\']} | {r[\'n_units\']} |\\")\\n        L += [\\"\\", \\"### ARTS vs each baseline (paired, per subject)\\", \\"\\",\\n              \\"| baseline | diff | 95% CI | p | wins | ties | losses |\\",\\n              \\"|---|---|---|---|---|---|---|\\"]\\n        wtl = {w[\\"baseline\\"]: w for w in d[\\"win_tie_loss\\"]}\\n        for c in d[\\"comparisons\\"]:\\n            w = wtl.get(c[\\"baseline\\"], {})\\n            L.append(f\\"| {c[\'baseline\']} | {c[\'diff_mean\']:+.2f} | \\"\\n                     f\\"[{c[\'diff_ci_low\']:+.2f}, {c[\'diff_ci_high\']:+.2f}] | \\"\\n                     f\\"{c[\'p\']:.4f} | {w.get(\'wins\',\'-\')} | \\"\\n                     f\\"{w.get(\'ties\',\'-\')} | {w.get(\'losses\',\'-\')} |\\")\\n        v = d[\\"verdict\\"]\\n        L += [\\"\\", f\\"**Verdict: {v[\'verdict\']}** -- {v[\'reason\']}. \\"\\n                  f\\"Binding comparison: {v[\'binding_comparison\']} \\"\\n                  f\\"({v[\'binding_diff\']:+.2f}, CI \\"\\n                  f\\"[{v[\'binding_ci\'][0]:+.2f}, {v[\'binding_ci\'][1]:+.2f}]).\\",\\n              \\"\\", \\"### Ablation\\", \\"\\",\\n              \\"| variant | acc mean | 95% CI |\\", \\"|---|---|---|\\"]\\n        for r in sorted(d[\\"ablation\\"], key=lambda x: x[\\"method\\"]):\\n            L.append(f\\"| {r[\'method\']} | {r[\'acc_mean\']:.2f} | \\"\\n                     f\\"[{r[\'acc_ci_low\']:.2f}, {r[\'acc_ci_high\']:.2f}] |\\")\\n        L += [\\"\\", \\"### Per subject (accuracy %)\\", \\"\\"]\\n        subs = sorted(d[\\"per_subject\\"])\\n        meths = [r[\\"method\\"] for r in d[\\"summary\\"]]\\n        L += [\\"| subject | \\" + \\" | \\".join(meths) + \\" |\\",\\n              \\"|\\" + \\"---|\\" * (len(meths) + 1)]\\n        for s in subs:\\n            L.append(f\\"| {s} | \\" + \\" | \\".join(\\n                f\\"{d[\'per_subject\'][s].get(m, float(\'nan\')):.1f}\\"\\n                for m in meths) + \\" |\\")\\n        L.append(\\"\\")\\n    return \\"\\\\n\\".join(L)\\n\\n\\nif __name__ == \\"__main__\\":\\n    import sys\\n    r = full_report(sys.argv[1], sys.argv[2] if len(sys.argv) > 2 else None)\\n    print(render_markdown(r))\\n"')
with open(f'{CODE_DIR}/analysis.py', 'w') as f:
    f.write(_src)
print('wrote analysis.py', len(_src), 'chars')


## Run

In [ ]:
# ---- run (resumable) ----------------------------------------------------
# Safe to re-run after a disconnect: completed units are skipped. Results are
# written after EVERY unit and fsync'd, so an interrupted session loses at
# most the unit in progress.
import importlib
import protocol as PR
importlib.reload(PR)

cfg = PR.RunCfg(
    ds1_dir=DS1_DIR, ds2a_dir=DS2A_DIR, out_dir=OUT_DIR,
    datasets=DATASETS, run_eegnet=RUN_EEGNET, run_ablation=RUN_ABLATION,
)
PR.run(cfg)


## Results

In [ ]:
# ---- analysis -----------------------------------------------------------
import importlib, json
import analysis as A
importlib.reload(A)

report = A.full_report(f"{OUT_DIR}/predictions.jsonl", OUT_DIR)
from IPython.display import Markdown, display
display(Markdown(A.render_markdown(report)))


In [ ]:
# ---- the pre-registered verdict, stated plainly -------------------------
for ds, d in report["datasets"].items():
    v = d["verdict"]
    print(f"\n=== {ds} ===")
    print(f"  VERDICT: {v['verdict']}")
    print(f"  reason : {v['reason']}")
    print(f"  binding comparison: ARTS vs {v['binding_comparison']}  "
          f"{v['binding_diff']:+.2f} pts  "
          f"CI [{v['binding_ci'][0]:+.2f}, {v['binding_ci'][1]:+.2f}]")
    print("  --- ablation (which component earns the gain) ---")
    for r in sorted(d["ablation"], key=lambda x: x["method"]):
        print(f"    {r['method']:22s} {r['acc_mean']:6.2f}")

if "ds1_real_subjects_only" in report:
    v = report["ds1_real_subjects_only"]["verdict"]
    print("\n=== ds1, four REAL subjects only (pre-specified secondary) ===")
    print(f"  VERDICT: {v['verdict']} ({v['reason']})")


## Integrity checks

In [ ]:
# ---- integrity checks ---------------------------------------------------
# These are cheap and catch the failure modes the audit found in the previous
# codebase: silently dropped methods, unequal unit counts, and duplicated rows.
import collections, json
recs = [json.loads(l) for l in open(f"{OUT_DIR}/predictions.jsonl") if l.strip()]

keys = [(r["dataset"], str(r["subject"]), r["seed"]) for r in recs]
dups = [k for k, c in collections.Counter(keys).items() if c > 1]
print("duplicate units:", len(dups), dups[:5])

per_method = collections.Counter()
for r in recs:
    for m, p in r["pred"].items():
        if p is not None:
            per_method[m] += 1
print("\nunits per method (these must all be equal):")
for m, c in sorted(per_method.items(), key=lambda kv: -kv[1]):
    print(f"   {m:22s} {c}")

skipped = collections.Counter()
for r in recs:
    for m, p in r["pred"].items():
        if p is None:
            skipped[m] += 1
print("\nexplicitly skipped (recorded, not silently dropped):", dict(skipped))

# every method must be scored on exactly the same test trials within a unit
bad = [k for r, k in zip(recs, keys)
       if any(len(p) != len(r["y_true"])
              for p in r["pred"].values() if p is not None)]
print("units with mismatched prediction lengths:", len(bad))
